# 03 · Planos mensuales, migraciones y trayectorias

> **Estado de evidencia:** los CSV suministrados son plantillas sin observaciones. Las celdas calculan resultados únicamente si se cargan extractos reales. No se fabrican hallazgos. Las fórmulas constituyen especificaciones metodológicas sujetas a aprobación del contrato de métricas.

**Convenciones:** $i$ crédito, $c$ cohorte de originación, $t$ corte calendario, $m$ meses desde originación (MOB), $D_{i,t}$ días de mora, $B_{i,t}$ saldo capital, $L_{i,t}$ cupo. Los límites de ventanas y las reglas de castigo deben parametrizarse.

## 1. Objetivo, alcance y decisiones

Se distinguen observaciones a nivel crédito-corte, crédito-MOB, evento y cliente. Se prohíben joins uno-a-muchos sin agregación previa. La trazabilidad debe conservar fecha de vigencia y disponibilidad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE = Path.cwd()
if not (BASE/'plantilla_panel_mensual.csv').exists():
    BASE = Path('/mnt/data/estudio_riesgo_credito')
assert BASE.exists(), 'Ubique el notebook junto a las plantillas CSV'
plt.rcParams.update({'figure.figsize':(9,4.8),'axes.grid':True,'grid.alpha':.2})
def load(name, parse_dates=None):
    path=BASE/name
    df=pd.read_csv(path,parse_dates=parse_dates or [])
    print(f'{name}: {len(df):,} registros, {len(df.columns)} columnas')
    return df

## Fotografía en el espacio saldo–DPD

Para cada corte $t$ el plano $\mathcal P_t=\{(B_{i,t},D_{i,t})\}_{i\in A_t}$ conserva observaciones individuales. Pueden añadirse tamaño de burbuja $L_{i,t}$ y color por segmento. El plano MOB $m$ agrupa distintas fechas calendario: **no es equivalente** al plano del corte $t$.

In [ ]:
df=load('plantilla_panel_mensual.csv')
if not df.empty:
    x=df.copy();x['mora']=pd.to_numeric(x.mora,errors='coerce');x['saldo_capital']=pd.to_numeric(x.saldo_capital,errors='coerce')
    cortes=sorted(x.fecha_corte.dropna().unique());t=cortes[-1]
    s=x[x.fecha_corte.eq(t)].dropna(subset=['mora','saldo_capital'])
    plt.scatter(s.saldo_capital,s.mora,alpha=.35,s=10)
    plt.axhline(30,ls='--',label='30 DPD');plt.xlabel('Saldo capital (unidad de extracto)');plt.ylabel('DPD');plt.title(f'Plano saldo–mora al corte {t}');plt.legend();plt.tight_layout();plt.show()
else: print('PENDIENTE: plano individual por corte')

## Matriz de transición y roll rates

Defina estados mutuamente excluyentes $S(D)$: $0$, $1$–$15$, $16$–$29$, $30$–$59$, $60$–$89$, $90+$. Para pares observados en $t,t+1$:
$$\widehat P_{ab}(t)=\frac{\sum_i\mathbf1(S_{i,t}=a,S_{i,t+1}=b)}{\sum_i\mathbf1(S_{i,t}=a)}.$$
Cada fila debe sumar uno (salvo estados sin observaciones). Los créditos sin observación en $t+1$ requieren estado de salida/censura, no eliminación silenciosa.

In [ ]:
if not df.empty:
    x=df.copy();x['mob']=pd.to_numeric(x.mob,errors='coerce');x['mora']=pd.to_numeric(x.mora,errors='coerce')
    bins=[-1,0,15,29,59,89,float('inf')];labels=['0','1-15','16-29','30-59','60-89','90+']
    x['estado']=pd.cut(x.mora,bins=bins,labels=labels)
    keys=['loan_key','merchant_key','corte_origen'];x=x.sort_values(keys+['mob'])
    x['estado_siguiente']=x.groupby(keys,observed=True).estado.shift(-1)
    x['mob_siguiente']=x.groupby(keys,observed=True).mob.shift(-1)
    pairs=x[x.mob_siguiente.eq(x.mob+1)]
    mat=pd.crosstab(pairs.estado,pairs.estado_siguiente,normalize='index').reindex(index=labels,columns=labels)
    display((mat*100).round(1))
    if mat.notna().any().any():
        plt.imshow(mat.fillna(0),vmin=0,vmax=1);plt.xticks(range(6),labels);plt.yticks(range(6),labels);plt.colorbar(label='Probabilidad');plt.xlabel('Estado t+1');plt.ylabel('Estado t');plt.title('Roll rates condicionados a observación consecutiva');plt.tight_layout();plt.show()
else: print('PENDIENTE: roll rates')

## Cura, deterioro y tiempo en mora

Para $k=30$, cura observada $C_{i,t}=\mathbf1(D_{i,t}\ge30,D_{i,t+1}<30)$; deterioro $W_{i,t}=\mathbf1(D_{i,t}<30,D_{i,t+1}\ge30)$. La tasa de cura condicional:
$$CR_{30,t}=\frac{\sum_i C_{i,t}}{\sum_i\mathbf1(D_{i,t}\ge30)}.$$
No atribuir la cura a una política sin control de confusión y censura.

In [ ]:
if not df.empty and len(pairs):
    pairs=pairs.copy();pairs['dpd_siguiente']=pairs.groupby(keys,observed=True).mora.shift(-1) if False else np.nan
    # Comparación por etiquetas, no imputar valores ausentes
    cure=(pairs.estado.isin(['30-59','60-89','90+']) & pairs.estado_siguiente.isin(['0','1-15','16-29'])).sum()
    at_risk=pairs.estado.isin(['30-59','60-89','90+']).sum()
    print('Cura condicional observada:',cure/at_risk if at_risk else 'denominador cero')
else: print('PENDIENTE: cura')

## Cadenas de Markov y límites

Bajo hipótesis de Markov homogénea $P(S_{t+h}=b\mid S_t=a)=(P^h)_{ab}$. La aproximación falla cuando la transición depende de MOB, segmento, condiciones macro o historia. Comparar $P_{m}$ por MOB, y contrastar probabilidades pronosticadas frente a frecuencias observadas.

In [ ]:
if not df.empty and len(pairs) and mat.notna().all().all():
    P=mat.to_numpy();print('Suma filas:',P.sum(axis=1).round(4));display(pd.DataFrame(np.linalg.matrix_power(P,3),index=labels,columns=labels).round(3))
else: print('PENDIENTE: matriz completa sin filas vacías')

## Limitaciones y siguiente ejecución

**Sin datos reales** no se pueden inferir tasas, significancia, calidad predictiva ni relaciones causales. Cargar las plantillas desde Athena con llaves y fechas; verificar los resultados de los controles antes de interpretar gráficos. Para producción: pruebas temporales, monitoreo de sesgo de selección, documentación de fuentes y revisión de privacidad.